# Satlas Super-Resolution Evaluation

This notebook compares one or more **super-resolution (SR) models**
against a ground-truth (GT) map using two metrics:

- **PSNR** — pixel-wise reconstruction fidelity.
- **CLIP image similarity** (ViT-B/32) — perceptual / semantic similarity.

Each model is evaluated at three granularities:

1. **Whole-image** — one score per model over the entire stitched map.
2. **Per-tile** — one score per cell of a `GRID_SIZE × GRID_SIZE` grid.
3. **Sliding patches** — overlapping `PATCH_GRID × PATCH_GRID` windows.

Results are aggregated into a single pandas DataFrame for easy comparison.

## What this notebook does *not* do

- **Stitching** of tile predictions — see `Tile_Stitching_Mapping.ipynb`.
- **Inference** — see `SSR_Inference.ipynb`.
- **Training / hyperparameter search** — see `SSR_Optuna.ipynb`.

## Prerequisites

1. A stitched GT map exists as a PNG (e.g. `GT_Summer.png` (Found in SatlasSuperResolution -> Summer -> Figures.)).
2. For each model to evaluate, a stitched SR map exists at the same
   geographic extent (different resolution is fine — GT is resized to
   match).
3. `torch`, `clip` (OpenAI), `opencv-python`, `pandas` and `Pillow` are
   installed.

In [ ]:
# Imports
from pathlib import Path
from typing import Callable

import clip
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

In [ ]:
# Configuration
# Adjust these paths to your local setup.

# Ground-truth stitched map (output of mapping.ipynb on GT tiles)
GT_PATH = Path('SatlasSuperResolution\Summer\Figures\GT_Summer.png')

# Stitched SR maps to evaluate. Key = model name (used in output table).
MODEL_MAPS: dict[str, Path] = {
    'ESRGAN': Path('SatlasSuperResolution\Summer\Figures\SSR_Summer.png'),
    'SwinIR': Path('SwinIR\Summer\Figures\SwinIR_Summer.png')}

# Grid that the stitched map was built from (cells per side).
GRID_SIZE = 17

# Sliding-patch evaluation: window of PATCH_GRID x PATCH_GRID cells, moved
# STRIDE_GRID cells at a time.
PATCH_GRID = 3
STRIDE_GRID = 1

# Where to save the aggregated results CSV.
RESULTS_CSV = Path('Evaluation_results.csv')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'GT     : {GT_PATH}')
print(f'Models : {list(MODEL_MAPS)}')

## Metric functions

PSNR is computed on raw uint8 BGR arrays. CLIP similarity uses the
ViT-B/32 image encoder; the model is loaded once and reused for every
comparison.

In [ ]:
def psnr(img1: np.ndarray, img2: np.ndarray) -> float:
    """Peak signal-to-noise ratio between two uint8 images of equal shape."""
    mse = np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    return 20.0 * np.log10(255.0 / np.sqrt(mse))


def load_clip_model(device: torch.device):
    """Load CLIP ViT-B/32 once; returns (model, preprocess)."""
    model, preprocess = clip.load('ViT-B/32', device=device)
    model.eval()
    return model, preprocess


def make_clip_similarity(model, preprocess, device: torch.device) -> Callable[[np.ndarray, np.ndarray], float]:
    """Return a function that computes CLIP cosine similarity between two BGR arrays."""

    def clip_similarity(img1_bgr: np.ndarray, img2_bgr: np.ndarray) -> float:
        pil1 = Image.fromarray(cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2RGB))
        pil2 = Image.fromarray(cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2RGB))
        t1 = preprocess(pil1).unsqueeze(0).to(device)
        t2 = preprocess(pil2).unsqueeze(0).to(device)
        with torch.no_grad():
            f1 = model.encode_image(t1)
            f2 = model.encode_image(t2)
            f1 = f1 / f1.norm(dim=-1, keepdim=True)
            f2 = f2 / f2.norm(dim=-1, keepdim=True)
            return (f1 @ f2.T).item()

    return clip_similarity


clip_model, clip_preprocess = load_clip_model(DEVICE)
clip_similarity = make_clip_similarity(clip_model, clip_preprocess, DEVICE)
print('CLIP ViT-B/32 loaded.')

## Image loading

The GT and SR maps usually do not share the same resolution (SR is 4x
the LR input; GT may be a different native resolution). The SR map is
treated as the reference resolution and GT is resized to match before
any metric is computed.

In [ ]:
def load_aligned_pair(pred_path: Path, gt_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load a prediction + GT image, resize GT to match the prediction."""
    pred = cv2.imread(str(pred_path))
    gt = cv2.imread(str(gt_path))
    if pred is None:
        raise FileNotFoundError(f'Could not read prediction: {pred_path}')
    if gt is None:
        raise FileNotFoundError(f'Could not read ground truth: {gt_path}')
    if gt.shape[:2] != pred.shape[:2]:
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))
    return pred, gt

## Evaluation routines

Three granularities, all returning a list of `(label, psnr, clip)`
records so they can be concatenated into one DataFrame at the end.

In [ ]:
def evaluate_whole(pred: np.ndarray, gt: np.ndarray) -> list[dict]:
    """Single PSNR + CLIP score over the whole image."""
    return [{
        'scope': 'whole',
        'label': 'full_map',
        'psnr': psnr(pred, gt),
        'clip': clip_similarity(pred, gt)}]


def evaluate_per_tile(pred: np.ndarray, gt: np.ndarray, grid_size: int) -> list[dict]:
    """Score every cell of a grid_size x grid_size grid."""
    H, W = pred.shape[:2]
    cell_h, cell_w = H // grid_size, W // grid_size
    records: list[dict] = []
    for gy in range(grid_size):
        for gx in range(grid_size):
            y0, y1 = gy * cell_h, (gy + 1) * cell_h
            x0, x1 = gx * cell_w, (gx + 1) * cell_w
            t_pred = pred[y0:y1, x0:x1]
            t_gt = gt[y0:y1, x0:x1]
            records.append({
                'scope': 'tile',
                'label': f'tile_{gx:02d}_{gy:02d}',
                'psnr': psnr(t_pred, t_gt),
                'clip': clip_similarity(t_pred, t_gt)})
    return records


def evaluate_patches(
    pred: np.ndarray,
    gt: np.ndarray,
    grid_size: int,
    patch_grid: int,
    stride_grid: int,
) -> list[dict]:
    """Score patches of patch_grid x patch_grid cells, stepped by stride_grid cells."""
    H, W = pred.shape[:2]
    cell_h, cell_w = H // grid_size, W // grid_size
    patch_h, patch_w = patch_grid * cell_h, patch_grid * cell_w
    stride_h, stride_w = stride_grid * cell_h, stride_grid * cell_w

    records: list[dict] = []
    patch_id = 0
    for y in range(0, H - patch_h + 1, stride_h):
        for x in range(0, W - patch_w + 1, stride_w):
            p_pred = pred[y:y + patch_h, x:x + patch_w]
            p_gt = gt[y:y + patch_h, x:x + patch_w]
            records.append({
                'scope': 'patch',
                'label': f'patch_{patch_id:03d}_x{x // cell_w}_y{y // cell_h}',
                'psnr': psnr(p_pred, p_gt),
                'clip': clip_similarity(p_pred, p_gt)})
            patch_id += 1
    return records


def evaluate_model(
    model_name: str,
    pred_path: Path,
    gt_path: Path,
    grid_size: int,
    patch_grid: int,
    stride_grid: int,
) -> pd.DataFrame:
    """Run all three evaluation scopes for one model and return a long-format DataFrame."""
    pred, gt = load_aligned_pair(pred_path, gt_path)
    print(f'\n[{model_name}] image shape: {pred.shape}')

    records: list[dict] = []
    records += evaluate_whole(pred, gt)
    records += evaluate_per_tile(pred, gt, grid_size)
    records += evaluate_patches(pred, gt, grid_size, patch_grid, stride_grid)

    df = pd.DataFrame(records)
    df.insert(0, 'model', model_name)
    return df

## Run evaluation

Loop over every model in `MODEL_MAPS`, compute all three scopes, and
concatenate the results into one long-format DataFrame.

In [ ]:
all_results = pd.concat(
    [
        evaluate_model(
            model_name=name,
            pred_path=path,
            gt_path=GT_PATH,
            grid_size=GRID_SIZE,
            patch_grid=PATCH_GRID,
            stride_grid=STRIDE_GRID)
        for name, path in MODEL_MAPS.items()],
    ignore_index=True)

all_results.to_csv(RESULTS_CSV, index=False)
print(f'\nSaved per-record results to: {RESULTS_CSV}')
all_results.head()

## Summary table

Aggregate the long-format results into a compact comparison: mean / min
/ max PSNR and CLIP per (model, scope). Infinite PSNR values (identical
patches) are excluded from PSNR aggregates.

In [ ]:
finite = all_results.replace([np.inf, -np.inf], np.nan)

summary = (
    finite
    .groupby(['model', 'scope'])
    .agg(
        psnr_mean=('psnr', 'mean'),
        psnr_min=('psnr', 'min'),
        psnr_max=('psnr', 'max'),
        clip_mean=('clip', 'mean'),
        clip_min=('clip', 'min'),
        clip_max=('clip', 'max'),
        n=('psnr', 'count'))
    .round(4))

summary

---